# Geração de Amostras V7 — Fase 2 (Alta Performance)

Este notebook executa o enriquecimento dos arquivos NPZ gerados na Fase 1, calculando os *soft-targets* (melhor jogada e scores de todas as posições) usando **Minimax com profundidade 7**.

### Otimizações Implementadas (Estratégias V2):
1. **Bitboard Engine**: O estado do tabuleiro é convertido de matrizes NumPy para um único inteiro de 64 bits. Operações de jogada e fechamento de caixa são feitas via álgebra booleana (AND/OR/SHIFT), eliminando loops lentos.
2. **Transposition Table (TT)**: Como ordens diferentes de traços podem levar ao mesmo tabuleiro, o motor armazena resultados já calculados em cache, podando bilhões de cálculos redundantes.
3. **Move Ordering**: O algoritmo testa primeiro as jogadas que fecham caixas, maximizando a eficiência da poda Alpha-Beta.
4. **Distribuição via Spark**: O cálculo é paralelizado entre todos os workers do cluster, garantindo que o processamento termine em horas, não dias.

**Diretório de Trabalho:** `/Workspace/Users/diondu@gmail.com/CNN/profundidade_7_v7_adaptativo`

In [ ]:
import os
import time
import json
import random
import numpy as np
from pathlib import Path
from typing import List, Tuple, Dict

# Configurações de Ambiente e Jogo (Tabuleiro 4x3)
INPUT_DIR = Path("/Workspace/Users/diondu@gmail.com/CNN/profundidade_7_v7_adaptativo")
ROWS, COLS = 4, 3
DEPTH_TARGET = 7
SCORE_INDISPONIVEL = -1_000_000_000.0

def build_bitboard_tables(rows, cols):
    """
    Pré-calcula as tabelas de consulta (lookup tables) para o motor de bits.
    Transforma as coordenadas da matriz em índices de bits (0-30).
    """
    h, w = 2 * rows + 1, 2 * cols + 1
    edge_to_bit = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                edge_to_bit[(r, c)] = bit_idx
                bit_to_label[bit_idx] = f"H_{r}_{c}" if r % 2 == 0 else f"V_{r}_{c}"
                bit_idx += 1
    
    n_edges = bit_idx
    # Máscaras das 12 caixas (cada uma composta por 4 bits específicos)
    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            mask = (1 << edge_to_bit[(r-1, c)]) | (1 << edge_to_bit[(r+1, c)]) | \
                   (1 << edge_to_bit[(r, c-1)]) | (1 << edge_to_bit[(r, c+1)])
            box_masks.append(mask)
            
    # Mapeia qual bit afeta quais caixas (máximo 2 caixas por traço)
    edge_boxes = [tuple(bm for bm in box_masks if bm & (1 << b)) for b in range(n_edges)]
    return n_edges, (1 << n_edges) - 1, tuple(box_masks), tuple(edge_boxes), bit_to_label

N_EDGES, ALL_MASK, BOX_MASKS, EDGE_BOXES, BIT_TO_LABEL = build_bitboard_tables(ROWS, COLS)
LABELS_CANONICOS = [BIT_TO_LABEL[i] for i in range(N_EDGES)]

In [ ]:
def solve_minimax_bitboard(edges, depth, alpha, beta, maximizing, tt):
    """
    Motor Minimax com Bitboard e Tabela de Transposição.
    O cálculo de fechamento de caixas é feito via Bitwise AND.
    """
    if depth == 0 or edges == ALL_MASK:
        return 0

    # Verificação na Tabela de Transposição (Cache)
    tt_key = (edges, depth, maximizing)
    if tt_key in tt:
        flag, val = tt[tt_key]
        if flag == 0: return val # Valor exato
        if flag == 1 and val >= beta: return val # Corte Beta
        if flag == 2 and val <= alpha: return val # Corte Alpha

    # Identifica jogadas e ordena por potencial de fechamento
    moves = []
    for i in range(N_EDGES):
        if not (edges & (1 << i)):
            closed = sum(1 for bm in EDGE_BOXES[i] if (edges | (1 << i)) & bm == bm)
            moves.append((i, closed))
    moves.sort(key=lambda x: x[1], reverse=True)
    
    orig_alpha = alpha
    best_val = -10000 if maximizing else 10000
    
    for bit, closed in moves:
        new_e = edges | (1 << bit)
        if maximizing:
            val = (closed + solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)) if closed > 0 else \
                  solve_minimax_bitboard(new_e, depth-1, alpha, beta, False, tt)
            best_val = max(best_val, val)
            alpha = max(alpha, best_val)
        else:
            val = (-closed + solve_minimax_bitboard(new_e, depth-1, alpha, beta, False, tt)) if closed > 0 else \
                  solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
            best_val = min(best_val, val)
            beta = min(beta, best_val)
        if beta <= alpha: break

    # Armazena na TT
    tt[tt_key] = (0 if best_val > orig_alpha and best_val < beta else (1 if best_val >= beta else 2), best_val)
    return best_val

In [ ]:
def processar_estado_spark(matriz_bytes):
    """
    Processa um único estado neutro vindo da Fase 1.
    Faz o mapeamento reverso Matriz -> Bitboard e inicia o Minimax.
    """
    mat = np.frombuffer(matriz_bytes, dtype=np.int8).reshape(9, 7)
    edges = 0
    idx = 0
    # Reconstrói o Bitboard a partir do encoding {0, 1, 8, 9}
    for r in range(9):
        for c in range(7):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                if mat[r, c] == 9: edges |= (1 << idx)
                idx += 1

    tt = {}
    scores = np.full(31, SCORE_INDISPONIVEL, dtype=np.float32)
    for i in range(31):
        if not (edges & (1 << i)):
            closed = sum(1 for bm in EDGE_BOXES[i] if (edges | (1 << i)) & bm == bm)
            new_e = edges | (1 << i)
            scores[i] = (closed + solve_minimax_bitboard(new_e, DEPTH_TARGET-1, -10001, 10001, True, tt)) if closed > 0 else \
                        solve_minimax_bitboard(new_e, DEPTH_TARGET-1, -10001, 10001, False, tt)
    
    # Define melhor jogada
    validos = [i for i, s in enumerate(scores) if s > -1e8]
    if not validos: return "", scores
    best_idx = random.choice([i for i in validos if scores[i] == np.max(scores[validos])])
    return BIT_TO_LABEL[best_idx], scores

In [ ]:
# Loop de Orquestração Spark
pendentes = sorted(INPUT_DIR.glob("dataset_pequeno_*.npz"))
estados_unicos = set()

print(f"Coletando estados únicos de {len(pendentes)} arquivos...")
for path in pendentes:
    with np.load(path) as d:
        if d["melhor_jogada"].size > 0 and d["melhor_jogada"][0] != "": continue
        for est in d["estados"]: estados_unicos.add(est.tobytes())

if estados_unicos:
    sc = spark.sparkContext
    lista_estados = list(estados_unicos)
    rdd = sc.parallelize(lista_estados, numSlices=sc.defaultParallelism * 10)
    
    print(f"Processando {len(lista_estados)} estados no cluster...")
    results = rdd.map(lambda b: (b, processar_estado_spark(b))).collect()
    cache_global = {b: res for b, res in results}
    
    # Atualização dos arquivos NPZ
    for path in pendentes:
        with np.load(path) as d:
            n = d["estados"].shape[0]
            mj_arr = np.empty(n, dtype="<U5")
            smj_arr = np.empty((n, 31), dtype=np.float32)
            for i in range(n):
                rotulo, scores = cache_global[d["estados"][i].tobytes()]
                mj_arr[i], smj_arr[i] = rotulo, scores
            
            np.savez_compressed(path, **{**d, "melhor_jogada": mj_arr, "score_melhor_jogada": smj_arr, "depth_melhor_jogada": np.full(n, 7, dtype=np.int8)})
        print(f"Arquivo {path.name} atualizado.")
else:
    print("Nenhum NPZ pendente encontrado.")